In [ ]:
# General imports
import time
from typing import List

# Data Processing
import pandas as pd
import numpy as np

# NBA API
from nba_api.stats.endpoints import leaguedashplayerstats, commonteamroster
from nba_api.stats.static import teams

# Statistics
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

# Plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Get raw data

In [ ]:
def get_player_stats(first_season: int, last_season: int) -> pd.DataFrame:
    assert first_season <= last_season, 'Last season must be later than first season'

    # Initialize a list to append data for each season
    list_seasons_data = []

    for season in range(first_season, last_season + 1):
        print(f"Fetching {season}...")

        # Initialize list to append data for each type of measure
        list_measure_data = []

        for measure_type in ['Base']:
            # Call API
            stats = leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star='Regular Season',
                measure_type_detailed_defense=measure_type
            )

            # Store the data
            df_stats = stats.get_data_frames()[0]

            # Drop _RANK columns
            # since they just represent position by different criteria
            for col in df_stats.columns:
                if "RANK" in col:
                    df_stats.drop(col, axis=1, inplace=True)

            # Drop the MIN column in Advanced and Scoring
            # because they are minutes per game instead of total minutes
            if measure_type != 'Base':
                df_stats.drop('MIN', axis=1, inplace=True)

            # Append data to list of dataframes for the season
            list_measure_data.append(df_stats)

            # Sleep to not overload API calls
            time.sleep(0.5)

        # Merge the data for the season
        df_season = list_measure_data[0]
        for df_extra in list_measure_data[1:]:
            df_season = df_season.merge(df_extra)

        # Add SEASON column to identify data when we concatenate all seasons together
        df_season['SEASON'] = season

        # Append data to the list of dataframes for all seasons
        list_seasons_data.append(df_season)

    # Concatenate the data together at set an ID for each player + season combination    
    df = pd.concat(list_seasons_data, ignore_index=True)
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    return df

In [ ]:
def get_player_positions(first_season: int, last_season: int) -> pd.DataFrame:
    all_teams = teams.get_teams()
    rosters = []

    for team in all_teams:
        for season in range(first_season, last_season + 1):
            roster = commonteamroster.CommonTeamRoster(
                team_id=team['id'],
                season=season
            )
            df_roster = roster.get_data_frames()[0]
            df_roster['SEASON'] = season

            rosters.append(roster.get_data_frames()[0])
            time.sleep(0.5)  # avoid rate limiting

    df = pd.concat(rosters)[['PLAYER_ID', 'SEASON', 'POSITION']]
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    return df[['ID', 'POSITION']]

In [ ]:
df_stats_raw = pd.read_csv("../assets/nba_player_stats.csv")
# df_stats_raw = get_player_stats(first_season=2016, last_season=2025)

In [ ]:
df_positions_raw = pd.read_csv("../assets/nba_player_positions.csv")
# df_positions_raw = get_player_positions(first_season=2016, last_season=2025)

In [ ]:
df_raw = df_stats_raw.merge(df_positions_raw, how='left')

# Data cleaning

In [ ]:
def clean_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    # Basketball statistics are highly dependent on sample size. When a player only plays 15 total minutes across a whole season, 
    # their advanced rates and percentages explode into unrealistic, hyper-volatile extremes.
    # We choose 250 minutes (equivalent to 10 games with 25 minutes of play)
    df = df[df['MIN'] >= 250]

    # Drop unneeded (this leaves only 'POSITION' as string column. All the remaining ones are numeric)
    df = df.drop(columns=['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'SEASON', 'TEAM_COUNT', 'GP', 'W', 'L', 'W_PCT', 'ID', 'AGE'])

    # Drop correlated
    corr_columns = [
        'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA',   # Redundant with shooting percentages
        'OREB', 'DREB',                               # Redundant with REB
        'PLUS_MINUS',                                 # Highly dependent on team quality
        'DD2', 'TD3',                                 # Double-doubles and triple-doubles are correlated with other metrics
        'NBA_FANTASY_PTS', 'WNBA_FANTASY_PTS'         # Computed directly from a formula using other stats
    ]
    df = df.drop(columns=corr_columns)

    # Normalize (per 36 minutes)
    # Totals suffer from volume bias if we don't normalize them. We follow the standard of giving stats per 36 minutes.
    for col in ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'BLKA', 'PF', 'PFD']:
        df[col] = df[col] / df['MIN'] * 36

    # Drop unneeded column after normalizing
    df = df.drop(columns='MIN')

    # Parse unlisted positions
    df['POSITION'] = df['POSITION'].fillna('not_listed')

    return df

In [ ]:
df = clean_raw_data(df_raw)

In [ ]:
df['POSITION'].value_counts()

# Experiment design

| Model Type                     | Feature Set (Skills Stats) |
|--------------------------------|-----------------------------|
| Supervised (Random Forest)     |	Evaluates how well on-court play style predicts a player's official traditional position |
| Unsupervised (K-Means)	     |  Discovers the "natural" functional roles in the modern NBA based purely on basketball utility |

1. The Splits

    df_pure_train (80% of pure players)

    df_pure_test (20% of pure players)

    df_clean = All 100% of df_pure + df_mixed + df_none

2. The Training Phase

    Supervised: Trains only on df_pure_train.

    Unsupervised: Trains blindly on df (asking for exactly 3 clusters).

3. The Final Evaluation & Comparison Phase

    Supervised Evaluation: Test it on df_pure_test to get your baseline classification accuracy (e.g., "Our classifier is 88% accurate at guessing traditional labels based on skill").

    The "Tweener" Analysis: Run your supervised model on df_mixed and df_none. (e.g., "The classifier forces 70% of G-F players into the 'Guard' bucket").

    The Unsupervised Mapping: Look at how the unsupervised model handled the whole league. (e.g., "Our skill-only clustering naturally split the league into Rim Protectors, Floor Spacers, and Playmakers").

    The Ultimate Showdown (The Comparison): Pull the df_pure_test rows from both models. Compare the Supervised Predictions against the Unsupervised Cluster Assignments using the Adjusted Rand Index (ARI) and a Confusion Matrix.

This final step answers your ultimate thesis question: When looking at a completely unbiased holdout set, do the natural, skill-based clusters mathematically resemble the NBA's traditional labels?

You have designed a brilliant pipeline. You are completely ready to drop this into Python, scale your features, and run the code!

# Supervised learning

## Data split

In [ ]:
df_pure = df[df['POSITION'].isin(['G', 'F', 'C'])].copy()
df_mixd = df[df['POSITION'].isin(['G-F', 'F-G', 'F-C', 'C-F'])].copy()
df_none = df[df['POSITION'].isna()].copy()

In [ ]:
def pure_position_train_test_split(
    df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Isolate features (X) from target (y)
    X = df.drop(columns='POSITION')
    y = df['POSITION']

    # Perform the stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20, 
        random_state=13,  # Ensures reproducible results
        stratify=y        # Stratify to avoid class imbalance
    )

    # Reconstruct df_pure_train df_pure_test dataframes
    df_train = X_train.copy()
    df_train['POSITION'] = y_train

    df_test = X_test.copy()
    df_test['POSITION'] = y_test

    return df_train, df_test

In [ ]:
df_pure_train, df_pure_test = pure_position_train_test_split(df_pure)

## Exploratory Data Analysis

In [ ]:
FEATURES = [
    'FG_PCT', 'FG3_PCT', 'FT_PCT', 'PTS',
    'REB', 'AST', 'STL', 'TOV',
    'BLK', 'BLKA', 'PF', 'PFD' 
]

In [ ]:
# Class distribution check
df_pure_train['POSITION'].value_counts()

In [ ]:
def plot_features_boxplots(df: pd.DataFrame, target: str='POSITION') -> None:
    
    # Initialize a 3x4 subplot grid
    fig = make_subplots(
        rows=3,
        cols=4,
        subplot_titles=FEATURES,  # Dynamic titles matching feature names
        vertical_spacing=0.08,    # Space out rows to keep text readable
        horizontal_spacing=0.06
    )

    # Iterate through features and map them to row/col indices
    for i, feature in enumerate(FEATURES):
        # Plotly grid indices are 1-based (not 0-based)
        row = (i // 4) + 1
        col = (i % 4) + 1
    
        # Add boxplot trace to specific grid coordinate
        fig.add_trace(
            go.Box(
                x=df[target],
                y=df[feature],
                name="",                    # Hides redundant individual names on x-axis
                boxpoints="outliers",       # Options: 'all', 'outliers', or False
                notched=False,
                marker_color="#1f77b4"    # Custom thematic styling choice
            ),
            row=row,
            col=col
        )

    # 3. Optimize layout configuration for complex grid densities
    fig.update_layout(
        height=800,
        width=1200,
        title_text="Statistical Profiling: 12-Feature Boxplot Matrix",
        title_x=0.5,                # Centered title
        showlegend=False,           # Disable unified legend since traces are self-evident
        template="plotly_dark"      
    )

    # Render interactive dashboard
    fig.show()

In [ ]:
plot_features_boxplots(df_pure_train)

The boxplot shows the statistical distribution of each feature with respect to each position. We can see that, in general, everything correlates to common knowledge:

- Centers play closer to the basket and therefore their `FG_PCT` is clearly higher. This is reversed when it comes to `FG3_PCT` because guards and forwards are much better at long-range shooting. Also `FT_PCT` gradually lowers from guards to centers as they're considered more accurate shooters. Nevertheless, when it comes `PTS` all positions are similar.

- `REB` clearly correlate to distance to the basket. On the other end `AST` shows that guards are better passers. In general, also, the furthest from the basket, the better a player is at collecting `STL`. When it comes to `TOV` there are slight differences but it is not a clear differentiator like other metrics.

- For `BLK` again centers dominate, although all positions receive a similar number of `BLKA`. Also the closer to the basket, the more `PF` one tends to make. However, all positions receive a similar number of `PFD`.

Overall we have several metrics that seem great predictors of the positions. However, we have to be careful with collinearity because some features might be giving redundant information if they are very correlated.

In [ ]:
df_pure_train.corr(numeric_only=True)

As it was already hinted there are some metrics that are essentially telling us the same story. E.g. in the group given by `FG_PCT`, `REB`, `BLK` there are high correlations and correspond to the idea "this player plays close to the basket and he is a center".

In order to asses the severity of these correlations we compute Variance Inflation Factors (VIF) which instead of looking at pairwise correlations it regresses each features against all the others to quantify how inflanted its variance is.

In [ ]:
def compute_vif(df: pd.DataFrame) -> pd.DataFrame:
    # Take the features
    dfn = df[FEATURES].copy()

    # Statsmodels VIF requires an explicit intercept (constant) to avoid forcing 
    # the regression through the origin (0,0), which severely inflates VIF math.
    dfn = add_constant(dfn)
    
    # Calculate VIF for each feature
    df_vif = pd.DataFrame()
    df_vif["feature"] = dfn.columns
    df_vif["VIF"] = [variance_inflation_factor(dfn.values, i) for i in range(dfn.shape[1])]
    df_vif = df_vif.query('feature != "const"')

    return df_vif.sort_values(by="VIF", ascending=False)

In [ ]:
compute_vif(df_pure_train)

Since all the VIFs are below 5, we will not drop any feature for our training.

In [ ]:
# Isolate features from target
X_train = df_pure_train[FEATURES]
y_train = df_pure_train['POSITION']

# ==========================================
# PATH A: The Supervised Model Pipeline
# ==========================================
# (Even if your chosen model doesn't strictly require scaling, 
# placing it here ensures uniform comparison architectures)
supervised_pipeline = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(random_state=8)
)

# Train the supervised model
supervised_pipeline.fit(X_train, y_train)

# Unsupervised learning

In [ ]:
df.describe().loc[['min', 'max']]

In [ ]:
# Isolate just the 16 numerical features
X_unsupervised = df[FEATURES].copy()

# 2. Build a dedicated unsupervised pipeline
unsupervised_pipeline = make_pipeline(
    StandardScaler(),  # Fits and transforms across the entire player landscape
    KMeans(n_clusters=3, random_state=8, n_init=10)
)

# 3. Generate your playstyle clusters
df['CLUSTER'] = unsupervised_pipeline.fit_predict(X_unsupervised)

In [ ]:
df

In [ ]:
def plot_clusters_against_target(df, position_col='POSITION', cluster_col='CLUSTER'):
    # Clean missing values and extract distinct categories safely
    temp_df = df.copy()

    # Define a clean, logical sequencing for basketball positions on the x-axis
    ideal_order = ['G', 'F', 'C', 'G-F', 'F-G', 'F-C', 'C-F', 'not_listed']
    # Filter order down to only what actually exists in the current data subset
    existing_positions = [pos for pos in ideal_order if pos in temp_df[position_col].unique()]

    # 2. Build and normalize the cross-tabulation matrix (Percentages per row)
    crosstab = pd.crosstab(temp_df[position_col], temp_df[cluster_col])
    crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

    # Reorder the rows to match our basketball layout sequence
    crosstab_pct = crosstab_pct.reindex(existing_positions)

    # 3. Initialize the Graph Objects Canvas
    fig = go.Figure()

    # Define a distinct color palette for the 3 clusters
    cluster_colors = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c'}

    # 4. Add each cluster as an independent stacked trace
    for cluster_id in sorted(df[cluster_col].unique()):
        # Pull percentages across positions for this specific cluster ID
        y_percentages = crosstab_pct[cluster_id].values

        fig.add_trace(
            go.Bar(
                x=existing_positions,
                y=y_percentages,
                name=f"Cluster {cluster_id}",
                marker_color=cluster_colors.get(cluster_id, '#7f7f7f'),
                hovertemplate="<b>Position:</b> %{x}<br>" +
                              "<b>Proportion:</b> %{y:.1f}%<br>" +
                              "<extra></extra>"
            )
        )

    # 5. Enforce stacking and styling properties
    fig.update_layout(
        title={
            'text': "Playstyle Cluster Distributions Across Listed Positions",
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title="Listed Position Profile",
        yaxis_title="Proportion of Position Category (%)",
        barmode='stack',
        template='plotly_dark',
        yaxis_ticksuffix='%',
        yaxis_range=[0, 100],
        legend_title_text="Assigned Cluster",
        height=550,
        width=850
    )
    
    fig.show()

In [ ]:
plot_clusters_against_target(df)

In [ ]:
plot_features_boxplots(df, target='CLUSTER')

In [ ]:
df['POSITION'].value_counts()

In [ ]:
df['CLUSTER'].value_counts().sort_index()

Cluster 2 quite clearly corresponds to centers. Cluster 1 corresponds to a mix position somewhat between G and F. Finally cluster 0 seems to be loosely associated to "pure guards" (possibly point guards). But

# Joint analysis

# Final comments

The threshold of $5$ represents a specific tipping point in the mathematical reliability of your model's stability:

- **The Variance Doubling Rule:** Mathematically, $\text{VIF} = \frac{1}{1 - R_i^2}$. A VIF of $5.0$ means that $80\%$ ($R^2 = 0.80$) of that feature's variance is perfectly explained by the other 15 features combined.

- **Standard Error Inflation:** The standard error of a feature's regression coefficient is multiplied by $\sqrt{\text{VIF}}$. At a VIF of $5.0$, your standard error is more than doubled ($\sqrt{5} \approx 2.24$).